In [1]:
from __future__ import annotations

import json
import os
from typing import Any

import psycopg
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from psycopg import sql

load_dotenv()

llm = ChatOpenAI(
    model="qwen3.8-flash",
    temperature=0,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
    extra_body={"enable_thinking": False},
)


c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### Direct agent

In [2]:
def get_connection():
    """Connect as read-only sql_agent; password from SQL_AGENT_PASSWORD in .env."""
    return psycopg.connect(
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        dbname=os.getenv("POSTGRES_DB", "tilik"),
        user="sql_agent",
        password=os.environ["SQL_AGENT_PASSWORD"],
        options="-c default_transaction_read_only=on -c statement_timeout=10000",
    )

ANALYTICS_TABLES = {
    "dataset", "measure", "period", "unit", "dimension",
    "dimension_member", "observation", "observation_dimension",
}

@tool
def list_tables() -> str:
    """List the analytics tables available for exploratory SQL."""
    return json.dumps(sorted(ANALYTICS_TABLES))

@tool
def describe_table(table_name: str) -> str:
    """Inspect the columns and a small sample of an allowed SQL table.

    Use this before writing exploratory SQL against an unfamiliar table.
    Return column names, SQL types, and at most five sample rows.
    This describes physical database structure; for a particular BPS
    dataset's measures and axes, use describe_dataset instead.

    Args:
        table_name: Exact name of a table allowed by list_tables.
    """
    if table_name not in ANALYTICS_TABLES:
        return json.dumps({"error": "Unknown analytics table"})

    with get_connection() as conn:
        cols = conn.execute(
            """
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public' AND table_name = %s
            ORDER BY ordinal_position
            """,
            (table_name,),
        ).fetchall()

        query = sql.SQL("SELECT * FROM {} LIMIT 5").format(
            sql.Identifier("public", table_name)
        )
        cursor = conn.execute(query)
        sample = cursor.fetchall()

    names = [name for name, _ in cols]
    return json.dumps({
        "columns": [{"name": name, "type": dtype} for name, dtype in cols],
        "sample": [dict(zip(names, row)) for row in sample],
    }, default=str)

@tool
def search_datasets(query: str) -> str:
    """Find BPS datasets relevant to a topic or table identifier.

    Search dataset titles and measure names; also accept an exact BPS
    table code if the implementation supports it. Return a short list of
    candidates with dataset.id, BPS table code, title, source/domain, and
    parser status. Use this when the relevant dataset ID is unknown.
    Search results are metadata and contain no published values.

    Args:
        query: A topic, title fragment, measure name, or BPS table code.
    """
    pattern = f"%{query.strip()}%"
    with get_connection() as conn:
        rows = conn.execute(
            """
            SELECT DISTINCT d.id, d.bps_table_id, d.tablesource,
                   d.domain_id, d.title, d.parser_status
            FROM dataset d
            LEFT JOIN measure m ON m.dataset_id = d.id
            WHERE d.title ILIKE %s
               OR m.original_name ILIKE %s
               OR m.canonical_name ILIKE %s
            ORDER BY d.title
            LIMIT 10
            """,
            (pattern, pattern, pattern),
        ).fetchall()

    return json.dumps([
        dict(zip(
            ["dataset_id", "bps_table_id", "tablesource",
             "domain_id", "title", "parser_status"],
            row,
        ))
        for row in rows
    ])

@tool
def describe_dataset(dataset_id: int) -> str:
    """Inspect the structure of one BPS dataset before retrieving values.

    Return its title and parser status; measures and units; available
    periods; and axes with exact member IDs, codes, and labels. Use this
    after search_datasets, or directly when the dataset ID is already
    known. Choose measure, period, and member filters from this result.
    This tool returns metadata, not numeric observations.

    Args:
        dataset_id: Integer dataset.id, not the BPS code or table title.
    """
    with get_connection() as conn:
        dataset = conn.execute(
            """
            SELECT id, bps_table_id, tablesource, domain_id,
                   title, parser_status, notes
            FROM dataset WHERE id = %s
            """,
            (dataset_id,),
        ).fetchone()
        if dataset is None:
            return json.dumps({"error": "Dataset not found"})

        measures = conn.execute(
            """
            SELECT m.id, m.original_name, m.definition,
                   m.source_note, u.source_name
            FROM measure m
            LEFT JOIN unit u ON u.id = m.source_unit_id
            WHERE m.dataset_id = %s
            ORDER BY m.id LIMIT 30
            """,
            (dataset_id,),
        ).fetchall()
        members = conn.execute(
            """
            SELECT d.id, d.source_code, d.original_label,
                   dm.id, dm.source_code, dm.original_label
            FROM dimension d
            JOIN dimension_member dm ON dm.dimension_id = d.id
            WHERE d.dataset_id = %s
            ORDER BY d.id, dm.id LIMIT 200
            """,
            (dataset_id,),
        ).fetchall()
        periods = conn.execute(
            """
            SELECT id, label FROM period
            WHERE dataset_id = %s ORDER BY id DESC LIMIT 50
            """,
            (dataset_id,),
        ).fetchall()

    return json.dumps({
        "dataset": dataset,
        "measures": measures,
        "axes_and_members": members,
        "periods": periods,
    }, default=str)

@tool
def get_observations(
    dataset_id: int,
    measure_id: int,
    member_ids: list[int],
    period_labels: list[str],
) -> str:
    """Retrieve published values for one dataset and measure.

    Return values with their periods, units, source text, status codes,
    and complete axis/member labels. Every ID in member_ids must match
    the same observation (AND semantics); an empty list selects all
    members. Period labels must exactly match those returned by
    describe_dataset; an empty list selects all periods. Report whether
    the response was truncated by its row limit. Do not calculate new
    totals or replace missing values with zero.

    Args:
        dataset_id: Integer dataset.id from search_datasets.
        measure_id: Measure ID belonging to that dataset.
        member_ids: Exact member IDs from describe_dataset.
        period_labels: Exact published period labels, such as ["2015", "2016"].
    """
    if len(member_ids) > 8 or len(period_labels) > 20:
        return json.dumps({"error": "Too many filters"})

    with get_connection() as conn:
        measure_ok = conn.execute(
            "SELECT 1 FROM measure WHERE id = %s AND dataset_id = %s",
            (measure_id, dataset_id),
        ).fetchone()
        if not measure_ok:
            return json.dumps({"error": "Measure is not in dataset"})

        selected_members = conn.execute(
            """
            SELECT d.id, dm.id
            FROM dimension_member dm
            JOIN dimension d ON d.id = dm.dimension_id
            WHERE d.dataset_id = %s AND dm.id = ANY(%s::int[])
            """,
            (dataset_id, member_ids),
        ).fetchall()
        if (len(selected_members) != len(set(member_ids))
                or len({axis_id for axis_id, _ in selected_members})
                   != len(selected_members)):
            return json.dumps({"error": "Invalid members or two members from one axis"})

        rows = conn.execute(
            """
            SELECT o.id, p.label, o.value_numeric, o.source_value_text,
                   o.status_code, u.source_name,
                   array_agg(
                       json_build_object(
                           'axis', d.original_label,
                           'member_id', dm.id,
                           'member', dm.original_label
                       ) ORDER BY d.id
                   ) AS members
            FROM observation o
            JOIN period p ON p.id = o.period_id
            LEFT JOIN unit u ON u.id = o.source_unit_id
            LEFT JOIN observation_dimension od ON od.observation_id = o.id
            LEFT JOIN dimension d ON d.id = od.dimension_id
            LEFT JOIN dimension_member dm ON dm.id = od.member_id
            WHERE o.dataset_id = %s AND o.measure_id = %s
              AND (%s::text[] IS NULL OR p.label = ANY(%s::text[]))
              AND NOT EXISTS (
                  SELECT 1 FROM unnest(%s::int[]) AS wanted(member_id)
                  WHERE NOT EXISTS (
                      SELECT 1 FROM observation_dimension match_od
                      WHERE match_od.observation_id = o.id
                        AND match_od.member_id = wanted.member_id
                  )
              )
            GROUP BY o.id, p.label, u.source_name
            ORDER BY p.label DESC, o.id
            LIMIT 101
            """,
            (dataset_id, measure_id,
             period_labels or None, period_labels or None, member_ids),
        ).fetchall()

    return json.dumps({
        "dataset_id": dataset_id,
        "measure_id": measure_id,
        "truncated": len(rows) > 100,
        "rows": rows[:100],
    }, default=str)

@tool
def run_query(query: str) -> str:
    """Run a Read-Only Postgres SQL query on the database."""
    if not query.strip().lower().startswith("select"):
        return "Error: Only SELECT queries are supported"
    try:
        with get_connection() as conn:
            cursor = conn.execute(query)
            columns = [col[0] for col in cursor.description]
            rows = cursor.fetchall()
            return json.dumps({
                "columns": columns,
                "rows": rows,
            })
    except Exception as e:
        return f"Error running query: {e}"

tools = [list_tables, describe_table, search_datasets, describe_dataset, get_observations, run_query]

SYSTEM_PROMPT = SystemMessage("""
You analyze BPS observations.
1. Search datasets by title or measure; choose a parsed dataset.
2. Describe it to identify measure, period, axis, and exact member IDs.
3. Use get_observations for ordinary value retrieval.
Each dataset is one BPS table. Do not combine measures or average published
percentages unless the question and available denominators justify it.
Treat totals as separate members. Report null values with their status.
Cite dataset identity, period, measure, members, and unit for numeric claims.
Distinguish source notes from your own interpretation.
""")

llm_with_tools = llm.bind_tools(tools)


def agent_node(state: MessagesState) -> dict[str, Any]:
    """Call the LLM; it may reply or request tool calls."""
    response = llm_with_tools.invoke([SYSTEM_PROMPT, *state["messages"]])
    return {"messages": [response]}


# --- Graph definition ---
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

sql_agent = graph.compile()

In [3]:
QUESTION = HumanMessage(
    content="Provinsi mana saja yang memiliki tiga TPT tertinggi pada Agustus 2025, dan berapa Angka Partisipasi Sekolah kelompok umur 16–18 tahun di masing-masing provinsi tersebut pada 2025?"
)

# Default LangGraph recursion_limit is 25 (agent <-> tools steps). Raise until tooling is tighter.
GRAPH_CONFIG = {"recursion_limit": 25}

step_messages: list[Any] = []
for step in sql_agent.stream(
    {"messages": [QUESTION]},
    config=GRAPH_CONFIG,
    stream_mode="updates",
):
    for node_name, update in step.items():
        print(f"\n=== node: {node_name} ===")
        for msg in update.get("messages", []):
            msg.pretty_print()
            step_messages.append(msg)

print("\n=== final answer ===")
print(step_messages[-1].content)


=== node: agent ===
================================== Ai Message ==================================

I'll start by finding the relevant datasets for unemployment rate (TPT) and school participation rate (APS).
Tool Calls:
  search_datasets (call_30e0cc1223d844619316a878)
 Call ID: call_30e0cc1223d844619316a878
  Args:
    query: Tingkat Pengangguran Terbuka TPT menurut provinsi
  search_datasets (call_144e20285149404a8ecea576)
 Call ID: call_144e20285149404a8ecea576
  Args:
    query: Angka Partisipasi Sekolah kelompok umur 16-18 tahun

=== node: tools ===
================================= Tool Message =================================
Name: search_datasets

[]
================================= Tool Message =================================
Name: search_datasets

[]

=== node: agent ===
================================== Ai Message ==================================

Let me try broader searches and check available tables.
Tool Calls:
  search_datasets (call_a15426be81ae4453a8a77446)
